In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression  ##

from sklearn import metrics
from sklearn.metrics import confusion_matrix,classification_report

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn import tree
# from warnings import filterwarnings
# filterwarnings("ignore")

##  Analysing Dataset


In [ ]:
df = pd.read_csv('data.csv',sep=',',encoding="utf-8")
df.head()

In [ ]:
df.info()

In [ ]:
df

In [ ]:
df.shape

In [ ]:
df.isna().sum()

In [ ]:
df.apply(lambda x:len(x.unique()))


In [ ]:
print('bp ',df['bp'].unique())
print('oxy ',df['oxy'].unique())
print('hb ',df['hb'].unique())
print('ecg ',df['ecg'].unique())
print('temp ',df['temp'].unique())
print('target ',df['target'].unique())


In [ ]:
plt.hist(df['hb'], bins=None, range=None, density=False)

In [ ]:
plt.hist(df['oxy'], bins=None, range=None, density=False)

In [ ]:
df.columns

In [ ]:
plt.figure(figsize=(20,10))
sns.countplot(x=df['temp'],data=df,hue='target',palette="twilight_shifted",saturation=2,dodge=True,)

plt.savefig("Target&temp.png")

In [ ]:
list = ['bp','oxy','hb','ecg','temp','target']
m=1
plt.figure(figsize=(20,15))
for i in list:
    plt.subplot(3,3,m)
    sns.countplot(x=i,data=df,hue='target',palette="twilight_shifted",
    saturation=2,dodge=True)
    m+=1
    
plt.savefig("TargetRelation.png")

In [ ]:
plt.figure(figsize=(10,10))
sns.heatmap(pd.DataFrame(df.corr()['target']).sort_values(by='target').transpose().drop('target',axis=1).transpose(),annot=True,cmap='twilight')
plt.savefig("TargetCorrelations.png")

In [ ]:
X = df.drop('target',axis=1)
Y = df['target']
from sklearn.feature_selection import SelectKBest, chi2
fs = SelectKBest(score_func=chi2, k='all')
fs.fit(X, Y)
per = []
for i in fs.scores_:
    per.append(round(((i/sum(fs.scores_))*100),3))

features_data = pd.DataFrame({'Feature':X.columns,'Scores':fs.scores_,'Importance (%)':per}).sort_values(by=['Scores'],ascending=False)

plt.figure(figsize=(9,4))
sns.barplot( 'Importance (%)','Feature',orient='h',data=features_data,palette='twilight_shifted_r')
insignificant = features_data.loc[features_data['Importance (%)']<0.005]['Feature'].unique()
features_data = features_data.set_index('Feature')
features_data
plt.savefig("FeatureImportance.png")

In [ ]:
plt.figure(figsize=(15,5))
plt.subplot(1,2,1)
sns.histplot(data=df,hue='target',x='temp',bins=20,element='poly')
plt.subplot(1,2,2)
sns.histplot(data=df,hue='target',x='ecg',bins=20,element='poly')
plt.savefig("Thalach&oldpeak_Histplot.png")

In [ ]:
plt.figure(figsize=(15,5))
plt.subplot(1,2,1)
sns.histplot(data=df,hue='target',x='oxy',bins=20,element='poly')
plt.subplot(1,2,2)
sns.histplot(data=df,hue='target',x='bp',bins=20,element='poly')
plt.savefig("Resting_blood_pressure&chol_Histplot.png")

In [ ]:
plt.figure(figsize=(15,10))
sns.heatmap(df.corr(),cbar = True, annot =True,cmap='twilight_shifted_r');
plt.savefig("heatmap.png")

In [ ]:
df.columns


In [ ]:
df.info()

In [ ]:
df.describe()

___________________________________

# Modelling
___________________________

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

from xgboost import XGBClassifier


from sklearn.ensemble import RandomForestClassifier
from sklearn import tree





from sklearn import metrics
from sklearn.metrics import confusion_matrix,classification_report
from sklearn.metrics import f1_score,plot_roc_curve,accuracy_score,roc_curve,roc_auc_score,recall_score
from mlxtend.plotting import plot_confusion_matrix

from sklearn.model_selection import GridSearchCV

from sklearn.ensemble import ExtraTreesClassifier

from sklearn.linear_model import RidgeCV
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline, make_union
from sklearn.preprocessing import PolynomialFeatures

from sklearn.ensemble import AdaBoostRegressor
from sklearn.feature_selection import SelectFwe, f_regression

In [ ]:
X = df.drop('target',axis=1).values
Y = df['target'].values

In [ ]:
df

In [ ]:
X.shape

In [ ]:
Y.shape

In [ ]:
# Split the dataset into training and testing.
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=0)

In [ ]:
print(X_train.shape)
print(X_test.shape)
print(Y_train.shape)
print(Y_test.shape)

------------------------
**` 1-LogisticRegression`**
----------------------

In [ ]:
# Train and predict.
LL = LogisticRegression(solver='liblinear',max_iter = 1000, random_state = 31)
LL.fit(X_train,Y_train)
Y_pred_lr = LL.predict(X_test) 
prd = LL.predict([[124,98,62,218,98]])
print("the prediciton is ",prd)
y_pred_lr = np.around(Y_pred_lr)
print(y_pred_lr)

In [ ]:
score_lr = round(accuracy_score(Y_pred_lr,Y_test)*100,4)
score_lr

In [ ]:
# Confusion matrix.
conf_mat = metrics.confusion_matrix(Y_test,Y_pred_lr)
plot_confusion_matrix(conf_mat,class_names=["1","2","3","4"],figsize=(12,5),cmap='copper_r'); #fn
plt.savefig("LRcrr.png")

In [ ]:
from sklearn import metrics
print(metrics.classification_report(Y_test,y_pred_lr))

-----------------------
**`-2- RandomForestClassifier`**
--------------------

In [ ]:
max_accuracy = 0


for x in range(500):
    rf = RandomForestClassifier(random_state=x)
    rf.fit(X_train,Y_train)
    Y_pred_rf = rf.predict(X_test)
    current_accuracy = round(accuracy_score(Y_pred_rf,Y_test)*100,2)
    if(current_accuracy>max_accuracy):
        max_accuracy = current_accuracy
        best_x = x

print(max_accuracy)
print(best_x)



In [ ]:
rf = RandomForestClassifier(random_state=best_x)
rf.fit(X_train,Y_train)
Y_pred_rf = rf.predict(X_test)
Y_pred_rf.shape


In [ ]:
score_rf = round(accuracy_score(Y_pred_rf,Y_test)*100,2)
score_rf

In [ ]:
Y_pred_rf = np.around(Y_pred_rf)
print(metrics.classification_report(Y_test,Y_pred_rf))

-----------------------
**`-3- XGBoost `**
--------------------

In [ ]:
xgb= XGBClassifier(loss='exponential', learning_rate=0.001, n_estimators=800, subsample=1.0, criterion='friedman_mse', 
                                  min_samples_split=2, 
                                  min_samples_leaf=5, min_weight_fraction_leaf=0.2, max_depth=20, min_impurity_decrease=0.1, 
                                  min_impurity_split=None, 
                                  init=None, random_state=None, max_features=None, verbose=1, max_leaf_nodes=None, warm_start=False, 
                                  presort='deprecated', 
                                  validation_fraction=0.2, n_iter_no_change=None, tol=0.0002)
                              
xgb.fit(X_train, Y_train)
Y_pred_xgb=xgb.predict(X_test)
score_xgb = round(accuracy_score(Y_pred_xgb,Y_test)*100,2)
score_xgb

In [ ]:
Y_pred_xgb = np.around(Y_pred_xgb)
print(metrics.classification_report(Y_test,Y_pred_xgb))

-----------------------
**`-4- Showing Final Scores`**
--------------------

In [ ]:
scores = [score_lr,score_rf,score_xgb]
Models = ["Logistic Regression","Random Forest","XGBoost"]    

for i in range(len(Models)):
    print("The accuracy score achieved using "+Models[i]+" is: "+str(scores[i])+" %")

In [ ]:
sns.set(style="darkgrid",rc={'figure.figsize':(20,10)})
plt.xlabel("Models",loc='center')
plt.ylabel("Accuracy score")

sns.barplot(Models,scores)
plt.savefig("AccuracyScores.png")

In [ ]:
df.describe()

In [ ]:
df.head()

In [ ]:
df.tail()


## Testing

In [ ]:
# Input = (125,96,58,202,98)
Input = (124,98,62,218,98)

Input_array= np.asarray(Input)
Input_reshaped = Input_array.reshape(1,-1)

prediction = LL.predict(Input_reshaped)
prediction = np.around(prediction)

print(prediction)

if (prediction[0]== 1):
  print('The Person does not have covid')
elif(prediction[0]== 2):
  print("The Person will has 1 %f "%(prediction))
elif(prediction[0]== 3):
    print("The Person will has 12%f "%(prediction))
elif(prediction[0]== 4):
    print("The Person will has 123 %f "%(prediction))


In [ ]:
Input = (172,71,87,847,107)

Input_array= np.asarray(Input)
Input_reshaped = Input_array.reshape(1,-1)

prediction = LL.predict(Input_reshaped)
prediction = np.around(prediction)

print(prediction)

if (prediction[0]== 1):
  print('The Person does not have issues')
elif(prediction[0]== 2):
  print("The Person will has 1 %f "%(prediction))
elif(prediction[0]== 3):
    print("The Person will has 12%f "%(prediction))
elif(prediction[0]== 4):
    print("The Person will has 123 %f "%(prediction))


In [ ]:
from tkinter import *
import tkinter as tk
# import tkinter as tk
# from tkinter import tk

import requests
from twilio.rest import Client



aaa=requests.get("https://api.thingspeak.com/channels/1720373/feeds.json?api_key=Z6PC42TJER35N0LL&results=2")
key="Z6PC42TJER35N0LL"
bp=aaa.json()['feeds'][-1]['field4']
oxy=aaa.json()['feeds'][-1]['field3']
hb=aaa.json()['feeds'][-1]['field1']
# ecg=aaa.json()['feeds'][-1]['field4']
tem=aaa.json()['feeds'][-1]['field2']

i=32
temp=str(i)
temp = (tem * 2)+temp
print(temp)

bp=str(bp)+'\t'  ##   +str(p)+'\n'
oxy=str(oxy)+'\t'
hb=str(hb)+'\t'
# ecg=str(ecg)+'\t'
temp=str(temp)+'\t'
print("\n Current bp feed is : \n\n"+str(bp))
print("\nCurrent oxy feed is: \n\n"+str(oxy))
print("\nCurrent hb feed is: \n\n"+str(hb))
# print("\nCurrent ecg feed is: \n\n"+str(ecg))
print("\nCurrent temp feed is: \n\n"+str(temp))










root = tk.Tk()
root.title('Remote patient monitoring')
root.geometry('850x650')
root.configure(background="#ffffff")
var = StringVar()
label = Label( root, textvariable = var,font=('arial',20,'bold'),bd=20,background="#ffffff")
var.set('patient survilence')  #########
label.grid(row=0,columnspan=6)

label_1 = ttk.Label(root, text ='Blood pressure',font=("Helvetica", 16),background="#ffffff")
label_1.grid(row=11,column=0)

label_11 = ttk.Label(root, text =bp,font=("Helvetica", 16),background="Purple3")
label_11.grid(row=11,column=1)
    
# Entry_1= Entry(root)
# Entry_1.grid(row=11,column=1)

label_2 = tk.Label(root, text ='SpO2',font=("Helvetica", 16),background="#ffffff")
label_2.grid(row=12,column=0)

label_22 = ttk.Label(root, text =oxy,font=("Helvetica", 16),background="Purple3")
label_22.grid(row=12,column=1)
    
# Entry_2 = Entry(root)
# Entry_2.grid(row=12,column=1)
    
    
label_3 = tk.Label(root, text ='Heart beat',font=("Helvetica", 16),background="#ffffff")
label_3.grid(row=13,column=0)

label_33 = ttk.Label(root, text =hb,font=("Helvetica", 16),background="Purple3")
label_33.grid(row=13,column=1)
    
# Entry_3 = Entry(root)
# Entry_3.grid(row=13,column=1)

label_4 = tk.Label(root, text ='ECG rate',font=("Helvetica", 16),background="#ffffff")
label_4.grid(row=14,column=0)

# label_44 = ttk.Label(root, text =ecg,font=("Helvetica", 16),background="Purple3")
# label_44.grid(row=14,column=1)
    
Entry_4= Entry(root)
Entry_4.grid(row=14,column=1)


label_5 = tk.Label(root, text ='Temperature',font=("Helvetica", 16),background="#ffffff")
label_5.grid(row=15,column=0)

label_55 = ttk.Label(root, text =temp,font=("Helvetica", 16),background="Purple3")
label_55.grid(row=15,column=1)
    
# Entry_5 = Entry(root)
# Entry_5.grid(row=15,column=1)


    
    
    
    

    


def fever():
    root1 = tk.Tk()
    root1.title('Remote patient Drug sugestion')
    root1.geometry('850x650')
    root1.configure(background="gray")
    label1=tk.Label(root1,text="Diagnosis Drugs for Fever",bg="pink",fg="black",font=("Arial",16),height=2,width=25)
    label1.place(x=50,y=30)
    label1=tk.Label(root1,text="Paracetamol",bg="pink",fg="black",font=("Arial",16),height=2,width=15)
    label1.place(x=100,y=100)
    label1=tk.Label(root1,text="acetaminophen",bg="pink",fg="black",font=("Arial",16),height=2,width=15)
    label1.place(x=100,y=200)
    label1=tk.Label(root1,text="Tylenol",bg="pink",fg="black",font=("Arial",16),height=2,width=15)
    label1.place(x=100,y=300)
    label1=tk.Label(root1,text="aspirin",bg="pink",fg="black",font=("Arial",16),height=2,width=15)
    label1.place(x=100,y=400)
    label1=tk.Label(root1,text="Acephen",bg="pink",fg="black",font=("Arial",16),height=2,width=15)
    label1.place(x=100,y=500)
    label1=tk.Label(root1,text="Ecpirin",bg="pink",fg="black",font=("Arial",16),height=2,width=15)
    label1.place(x=100,y=600)
    
    
    
def chest_pain():
    root2 = tk.Tk()
    root2.title('Remote patient Drug sugestion')
    root2.geometry('850x650')
    root2.configure(background="gray")
    label1=tk.Label(root2,text="Diagnosis Drugs for chest_pain",bg="pink",fg="black",font=("Arial",16),height=2,width=25)
    label1.place(x=50,y=30)
    label1=tk.Label(root2,text="Amlodipine",bg="pink",fg="black",font=("Arial",16),height=2,width=15)
    label1.place(x=100,y=100)
    label1=tk.Label(root2,text="Nadroparin",bg="pink",fg="black",font=("Arial",16),height=2,width=15)
    label1.place(x=100,y=200)
    label1=tk.Label(root2,text="Isosorbide",bg="pink",fg="black",font=("Arial",16),height=2,width=15)
    label1.place(x=100,y=300)
    label1=tk.Label(root2,text="Nifedipine",bg="pink",fg="black",font=("Arial",16),height=2,width=15)
    label1.place(x=100,y=400)
    label1=tk.Label(root2,text="Atenolol",bg="pink",fg="black",font=("Arial",16),height=2,width=15)
    label1.place(x=100,y=500)
    label1=tk.Label(root2,text="Diltiazem",bg="pink",fg="black",font=("Arial",16),height=2,width=15)
    label1.place(x=100,y=600)
    
def viral_fever():
    root3 = tk.Tk()
    root3.title('Remote patient Drug sugestion')
    root3.geometry('850x650')
    root3.configure(background="gray")
    label1=tk.Label(root3,text="Diagnosis Drugs for viral_fever",bg="pink",fg="black",font=("Arial",16),height=2,width=25)
    label1.place(x=50,y=30)
    label1=tk.Label(root3,text="Tylenol",bg="pink",fg="black",font=("Arial",16),height=2,width=15)
    label1.place(x=100,y=100)
    label1=tk.Label(root3,text="acetaminophen",bg="pink",fg="black",font=("Arial",16),height=2,width=15)
    label1.place(x=100,y=200)
    label1=tk.Label(root3,text="ibuprofen",bg="pink",fg="black",font=("Arial",16),height=2,width=15)
    label1.place(x=100,y=300)
    label1=tk.Label(root3,text="paracetamol",bg="pink",fg="black",font=("Arial",16),height=2,width=15)
    label1.place(x=100,y=400)
    label1=tk.Label(root3,text="Advil",bg="pink",fg="black",font=("Arial",16),height=2,width=15)
    label1.place(x=100,y=500)
    label1=tk.Label(root3,text=" Motrin IB",bg="pink",fg="black",font=("Arial",16),height=2,width=15)
    label1.place(x=100,y=600)
    


def predict():
    bp =  Entry_1.get()
    oxy =  Entry_2.get()
    hb =  Entry_3.get()
    ecg =  Entry_4.get()
    temp =  Entry_5.get()
       
    out = rf.predict([[float(bp),
       float(oxy),
       float(hb),
       float(ecg),
       float(temp)]])     ##float(area)
    
    output.delete(0,END)
    output.insert(0,out[0])
    
    print(out)
    if out[0] == 1 :
        output.delete(0,END)
        output.insert(0,'Normal')
        
    elif out[0] == 2: 
        output.delete(0,END)
        output.insert(0,'Fever')
        b3=tk.Button(root,text="fever",bg="pink",fg="black",font=("Arial",16),width=10,command=fever)
        b3.place(x=150,y=400)
        
    elif out[0] == 3:
        output.delete(0,END)
        output.insert(0,'Chest pain')
        b3=tk.Button(root,text="chest_pain",bg="pink",fg="black",font=("Arial",16),width=10,command=chest_pain)
        b3.place(x=150,y=400)
        
    elif out[0] == 4:
        output.delete(0,END)
        output.insert(0,'Viral fever')
        b3=tk.Button(root,text="viral_fever",bg="pink",fg="black",font=("Arial",16),width=10,command=viral_fever)
        b3.place(x=150,y=400)
        

    lbp=int(float(bp))
    print(lbp)
    if (0 < lbp <90):
        label_lbp = tk.Label(root, text ='Low Blood pressure',font=("Helvetica", 16),background="#ffffff")
        label_lbp.grid(row=22,column=1)
    elif (180 < lbp <230):
        label_hbp = tk.Label(root, text ='High Blood Pressure',font=("Helvetica", 16),background="#ffffff")
        label_hbp.grid(row=22,column=1)
    elif (91 < lbp <179):
        label_hbp = tk.Label(root, text ='Normal  Blood Pressure',font=("Helvetica", 16),background="#ffffff")
        label_hbp.grid(row=22,column=1)
        
        
    if out[0] == 4 :
        label_co = tk.Label(root, text ='Chance of covid',font=("Helvetica", 16),background="#ffffff")
        label_co.grid(row=23,column=1)
#     if out[0] == 2:
#         b2 = Button(root, text = 'Diagnosis',font=("Helvetica", 16),background="#5e6166",command=fever)
#         b2.grid(row=25,column=0)
#     if out[0] == 3:
#         b2 = Button(root, text = 'Diagnosis',font=("Helvetica", 16),background="#5e6166",command=chest_pain)
#         b2.grid(row=25,column=0)
#     if out[0] == 4:
#         b2 = Button(root, text = 'Diagnosis',font=("Helvetica", 16),background="#5e6166",command=viral_fever)
#         b2.grid(row=25,column=0)
        
  
#         # Your Account Sid and Auth Token from twilio.com / console
#         account_sid = 'AC55e8e4597d26f745e73359f5ecaf768d'
#         auth_token = '7ed023015709c75641e97c39e69f626d'

#         client = Client(account_sid, auth_token)

#         ''' Change the value of 'from' with the number 
#         received from Twilio and the value of 'to'
#         with the number in which you want to send message.'''
#         message = client.messages.create(
#                                       from_='+13254201666',
#                                       body ='body',
#                                       to ='+15558675310'
#                                   )

#         print(message.sid)
        
    
    
    
# bp =  Entry_1.get()
# oxy =  Entry_2.get()
# hb =  Entry_3.get()
# ecg =  Entry_4.get()
# temp =  Entry_5.get()
#     b2 = Button(root, text = 'Diagnosis',font=("Helvetica", 16),background="#ffffff",command=diagnosis)
#     b2.grid(row=25,column=0)


b1 = Button(root, text = 'predict',font=("Helvetica", 16),background="#cfb6fc",command = predict)
b1.grid(row=19,column=0)
    

output = Entry(root)
output.grid(row=19,column=1)
    
root.mainloop()


In [ ]:
type(temp)

In [ ]:
# Input = (124,98,62,218,98)  normal

In [ ]:
# 130	93	68	453	100	   2  fever
# 141	86	78	603	101	   3  chest pain
#  170	60	86	846	108    4 viral fever
